# 03 Logistic Regression

We train models on the training students and compare them on the validation students.
The test set is not used here. It is used only once, at the very end.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss

from src.data import load_data, split_students
from src.features import build_features

df = build_features(split_students(load_data()))
df["log_n_prev"] = np.log1p(df["n_prev"])

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]
print("train:", len(train), " val:", len(val))

train: 977349  val: 200546


## 1. Baselines
Simple guesses that any real model must beat.
- **Majority:** always predict "correct".
- **Question difficulty:** predict each question's training correct rate.

In [2]:
results = []

def score(name, y_true, prob):
    """AUC and accuracy for one predictor on validation."""
    results.append({
        "model": name,
        "val AUC": round(roc_auc_score(y_true, prob), 4),
        "val accuracy": round(accuracy_score(y_true, prob >= 0.5), 4),
    })

score("Majority (always correct)", val["IsCorrect"], np.ones(len(val)))
score("Question difficulty", val["IsCorrect"], val["q_correct_rate"])
pd.DataFrame(results)

,model,val AUC,val accuracy
0,Majority (always correct),0.5000,0.5472
1,Question difficulty,0.6754,0.6302


**Takeaway:** Always guessing "correct" gives 54.7% accuracy on validation (AUC 0.5, since it gives every answer the same score and can't rank). Question difficulty alone reaches AUC 0.675 and 63.0% accuracy. Our models must beat these.

## 2. Logistic regression
Three basic features: the student's accuracy so far, how many answers they have given (log), and the question's correct rate.
The scaler is fitted on training data only.

In [3]:
base_features = ["acc_prev", "log_n_prev", "q_correct_rate"]

logreg = make_pipeline(StandardScaler(), LogisticRegression())
logreg.fit(train[base_features], train["IsCorrect"])

val_prob = logreg.predict_proba(val[base_features])[:, 1]
score("Logistic regression (base)", val["IsCorrect"], val_prob)
pd.DataFrame(results)

,model,val AUC,val accuracy
0,Majority (always correct),0.5000,0.5472
1,Question difficulty,0.6754,0.6302
2,Logistic regression (base),0.7718,0.7041


### What did the model learn?
Each weight shows how much a feature pushes the prediction toward "correct". Features are scaled, so weights can be compared.

In [4]:
weights = pd.Series(logreg[-1].coef_[0], index=base_features).sort_values()
weights.round(3)

log_n_prev        0.028
q_correct_rate    0.733
acc_prev          0.855
dtype: float64

**Takeaway:** With three features, logistic regression reaches val AUC 0.772 and 70.4% accuracy, clearly beating both baselines. Student accuracy so far (0.855) and question correct rate (0.733) carry almost all the weight. The number of earlier answers adds almost nothing (0.028) once accuracy is known.

## 3. Forward selection: do topic clues help?

We start from the 3 base features and try adding each topic level (its accuracy so far and log answer count).
At each step we keep the level that raises validation AUC the most.
We stop when the best addition improves AUC by less than 0.001.

In [5]:
# log of the answer counts for each topic level, like log_n_prev
for lv in ["L1", "L2", "L3"]:
    df["log_n_prev_" + lv] = np.log1p(df["n_prev_" + lv])

train = df[df["split"] == "train"]
val = df[df["split"] == "val"]


def val_auc(features):
    """Train logistic regression on train with these features, return val AUC."""
    model = make_pipeline(StandardScaler(), LogisticRegression())
    model.fit(train[features], train["IsCorrect"])
    prob = model.predict_proba(val[features])[:, 1]
    return roc_auc_score(val["IsCorrect"], prob)

In [6]:
base_features = ["acc_prev", "log_n_prev", "q_correct_rate"]
candidates = {
    "L1": ["acc_prev_L1", "log_n_prev_L1"],
    "L2": ["acc_prev_L2", "log_n_prev_L2"],
    "L3": ["acc_prev_L3", "log_n_prev_L3"],
}
MIN_GAIN = 0.001

selected = list(base_features)
best_auc = val_auc(selected)
log = [{"step": 0, "tried": "(base only)", "val AUC": round(best_auc, 4), "kept": True}]

step = 1
while candidates:
    # try adding each remaining level
    tried = {}
    for level, cols in candidates.items():
        tried[level] = val_auc(selected + cols)
        log.append({"step": step, "tried": "+ " + level, "val AUC": round(tried[level], 4), "kept": False})

    # the best one this step
    best_level = max(tried, key=tried.get)
    gain = tried[best_level] - best_auc

    if gain < MIN_GAIN:
        print(f"Step {step}: best is {best_level} (+{gain:.4f}), below {MIN_GAIN}. Stop.")
        break

    # keep it
    selected += candidates.pop(best_level)
    best_auc = tried[best_level]
    log[-len(tried) + list(tried).index(best_level)]["kept"] = True
    print(f"Step {step}: add {best_level} (+{gain:.4f}), val AUC now {best_auc:.4f}")
    step += 1

print("\nSelected features:", selected)
pd.DataFrame(log)

Step 1: add L2 (+0.0045), val AUC now 0.7763


Step 2: best is L1 (+0.0006), below 0.001. Stop.

Selected features: ['acc_prev', 'log_n_prev', 'q_correct_rate', 'acc_prev_L2', 'log_n_prev_L2']


,step,tried,val AUC,kept
0,0,(base only),0.7718,True
1,1,+ L1,0.7747,False
2,1,+ L2,0.7763,True
3,1,+ L3,0.7755,False
4,2,+ L1,0.7769,False
5,2,+ L3,0.7767,False


**Takeaway:** Starting from val AUC 0.7718 with the base features, forward selection added level-2 skill history (+0.0045, to 0.7763).

In step 1 every level helped a little (L1 +0.0029, L3 +0.0037), but L2 helped most: L1 mostly repeats overall accuracy, and L3 topics often have too little history.

After L2, the best addition was L1 (+0.0006), below our 0.001 threshold, so we stopped.

Final features: acc_prev, log_n_prev, q_correct_rate, acc_prev_L2, log_n_prev_L2. We use these same features for the neural network.

Topic history adds about 0.005 AUC. Most of the model's power comes from student ability and question difficulty.

## 4. Tuning C (regularization)

`C` controls how strongly the model is kept simple: a small C keeps the weights small (less overfitting), a large C lets them grow freely.
We try C from 0.0001 to 100 with the selected features and keep the smallest C with the best validation log loss. We use log loss because our project depends on good probabilities, not just ranking.

In [7]:
final_features = list(selected)   # the features forward selection picked

rows = []
for C in [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]:
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C))
    model.fit(train[final_features], train["IsCorrect"])
    prob = model.predict_proba(val[final_features])[:, 1]
    rows.append({"C": C,
                 "val AUC": round(roc_auc_score(val["IsCorrect"], prob), 5),
                 "val log loss": round(log_loss(val["IsCorrect"], prob), 5)})

tuning = pd.DataFrame(rows)
best_C = tuning[tuning["val log loss"] == tuning["val log loss"].min()]["C"].min()
print("Features:", final_features)
print("Chosen C:", best_C)
tuning

Features: ['acc_prev', 'log_n_prev', 'q_correct_rate', 'acc_prev_L2', 'log_n_prev_L2']
Chosen C: 0.01


,C,val AUC,val log loss
0,0.0001,0.77626,0.56763
1,0.0010,0.77630,0.56714
2,0.0100,0.77630,0.56711
3,0.1000,0.77630,0.56711
4,1.0000,0.77630,0.56711
5,10.0000,0.77630,0.56711
6,100.0000,0.77630,0.56711


**Takeaway:** Validation AUC is the same (0.7763) for every C from 0.001 to 100, and only C = 0.0001 is slightly worse. With ~977,000 training answers and 5 features the model has nothing to overfit, so regularization barely matters. C = 0.01 is the smallest value with the best log loss (0.5671), so we use it.

## 5. Final logistic regression
The selected features and C, trained on the training students only.

In [8]:
final_model = make_pipeline(StandardScaler(), LogisticRegression(C=best_C))
final_model.fit(train[final_features], train["IsCorrect"])

val_prob = final_model.predict_proba(val[final_features])[:, 1]
score("Logistic regression (final)", val["IsCorrect"], val_prob)
pd.DataFrame(results)

,model,val AUC,val accuracy
0,Majority (always correct),0.5000,0.5472
1,Question difficulty,0.6754,0.6302
2,Logistic regression (base),0.7718,0.7041
3,Logistic regression (final),0.7763,0.7084


In [9]:
final_weights = pd.Series(final_model[-1].coef_[0], index=final_features).sort_values()
final_weights.round(3)

log_n_prev_L2    -0.025
log_n_prev        0.043
acc_prev_L2       0.283
acc_prev          0.648
q_correct_rate    0.731
dtype: float64

**Takeaway:** The final model reaches val AUC 0.7763 and 70.8% accuracy. Question correct rate is now the largest weight (0.731). Student ability is split between overall accuracy (0.648) and level-2 skill accuracy (0.283): the two are correlated, so they share the credit, and together they say a bit more than overall accuracy alone. The answer counts stay near zero.

## 6. Save predictions
We save predictions for validation and test so the evaluation notebook can compare all models without retraining.
We do **not** score the test set here. It is scored once, in the evaluation notebook.

In [10]:
to_predict = df[df["split"].isin(["val", "test"])]

preds = pd.DataFrame({
    "AnswerId": to_predict["AnswerId"],
    "split": to_predict["split"],
    "p_correct": final_model.predict_proba(to_predict[final_features])[:, 1],
})
preds.to_csv("../results/preds_logreg.csv", index=False)
print(preds.shape)
preds.head()

(405378, 3)


,AnswerId,split,p_correct
333,1160716,val,0.712561
334,1114855,val,0.762570
335,376576,val,0.901669
336,968084,val,0.757828
337,185605,val,0.883336


**Takeaway:** Saved 405,378 predictions (200,546 validation + 204,832 test) to `results/preds_logreg.csv` with columns AnswerId, split, p_correct. The neural network saves the same format, so the evaluation notebook can compare the models answer by answer.